# 01 Symbol Backtest

This notebook backtests one symbol in detail. The main workflow lives in the **interactive control panel** in Cell 4.

Recommended flow:

1. Run Cell 1 through Cell 4.
2. Select symbol, account mode, date range, and max bars.
3. Click **Run selected symbol** to render the full report.
4. Click **Compare selected symbols** to compare several symbols using the same date window and active configuration.

The cells after Cell 4 are optional: replay, Monte Carlo, and export.


In [ ]:
#

import sys
from pathlib import Path


def _find_root(start: Path, marker: str = 'pyproject.toml') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError(f'Could not find repo root containing {marker!r} and core_python/shared')


ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)

In [ ]:
#

from IPython.display import display

from core_python.shared.monte_carlo import plot_monte_carlo, run_monte_carlo
from core_python.strategies.combo.params import SYMBOLS, summary as strategy_summary
from core_python.strategies.combo.research_utils import (
    build_symbol_backtest_widget,
    build_symbol_replay_backtest_widget,
    build_symbol_replay_widget,
    configure_notebook,
    export_result_bundle,
    export_ctrader_validation_bundle,
    show_run_config,
)
from core_python.strategies.combo.symbol.backtest import run_symbol_backtest

configure_notebook()
print(strategy_summary())
print('Symbols:', ', '.join(SYMBOLS.keys()))


In [ ]:
#

RUN_CONFIG = {
    'symbol': 'US30',
    'account_mode': 'standard',
    'initial_balance': 100_000.0,
    'date_from': '2023-01-01',
    'date_to': None,
    'max_bars': 50_000,
    'indicator_overrides': {},
    'symbol_overrides': {},
    'collect_events': True,
    'replay': {
        'lookback_bars': 50,
        'step_bars': 1,
        'default_speed_ms': 800,
        'start_at_end': False,
        'show_signals': True,
        'show_entries': True,
        'show_exits': True,
    },
    'export_report': False,
    'export_ctrader': False,
}

show_run_config('Default Configuration', RUN_CONFIG)


In [ ]:
#
# Single-symbol backtest:
#
# Compare symbols:

symbol_dashboard = build_symbol_backtest_widget(
    symbols=SYMBOLS,
    run_symbol_backtest=run_symbol_backtest,
    default_config=RUN_CONFIG,
    global_ns=globals(),
)
display(symbol_dashboard)

In [ ]:
# Cell 6 - Interactive replay backtest dashboard
#
# Choose the symbol and strategy parameters, run a fresh Combo backtest, then
# replay the actual engine events. This is a research view only; it does not
# change live execution behavior.

symbol_replay_dashboard = build_symbol_replay_backtest_widget(
    symbols=SYMBOLS,
    run_symbol_backtest=run_symbol_backtest,
    default_config=RUN_CONFIG,
    global_ns=globals(),
)
display(symbol_replay_dashboard)


In [ ]:
# Cell 6 - Monte Carlo robustness for the latest result

if 'result' not in globals():
    print('No result yet. Click Run selected symbol in Cell 4 first.')
elif not result.trades:
    print('Skipping Monte Carlo because the current result has no trades.')
else:
    trade_pnls = [float(t['pnl_usd']) for t in result.trades]
    mc = run_monte_carlo(
        trade_pnls,
        n_iter=500,
        dd_threshold=0.20,
        initial_balance=RUN_CONFIG['initial_balance'],
    )
    print('Monte Carlo P(max DD > 20%) =', round(mc['prob_exceed_dd'] * 100, 2), '%')
    print('Sharpe CI 95% =', (round(mc['sharpe_ci_low'], 2), round(mc['sharpe_ci_high'], 2)))
    plot_monte_carlo(mc)


In [ ]:
# Cell 7 - Optional result export

EXPORT_REPORT = False
if 'result' not in globals():
    print('No result yet. Click Run selected symbol in Cell 4 first.')
elif EXPORT_REPORT:
    out = export_result_bundle(
        f"{RUN_CONFIG['symbol']}_{RUN_CONFIG['account_mode']}_symbol_backtest",
        metrics=result.metrics,
        trades=result.trades,
        equity=result.equity,
    )
    print('Exported:', out)
else:
    print('Export is disabled. Set EXPORT_REPORT = True to save CSV files.')


In [ ]:
# Cell 8 - Optional cTrader validation export

if 'result' not in globals():
    print('No result yet. Click Run selected symbol in Cell 4 first.')
elif RUN_CONFIG.get('export_ctrader'):
    out = export_ctrader_validation_bundle(
        result,
        RUN_CONFIG,
        name=f"{result.symbol}_{RUN_CONFIG['account_mode']}_symbol_backtest",
    )
    print('cTrader validation export:', out)
else:
    print("cTrader export is disabled. Set RUN_CONFIG['export_ctrader'] = True to save validation CSV files.")
